# A masked diffusion language model, trained on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/tinystories-ARM-model/blob/main/diffusion_models/diffusion_colab.ipynb)

An autoregressive model writes left to right, one token per step. A **masked diffusion model** starts from a
fully masked text and fills it in over a fixed number of steps, the positions it's most sure of first. Training
is the reverse: hide a random fraction `t/T` of a real text's tokens behind `[MASK]` and learn to predict them,
looking both ways.

The code is in `diffusion_models/` (`config`, `data`, `model`, `noising`, `train`, `sample`, `render`); the
first version is kept unchanged in `diffusion_models/v1_original.py`. It uses GPT-2's tokenizer and the same
token files as the autoregressive models, so the two can be compared on the same data.

| Step | What happens |
|---|---|
| 0 | Get the code |
| 1 | Settings: run mode, dataset, Google Drive |
| 2 | Tokenize the dataset (or copy the tokens from Drive) |
| 3 | Look at the noising |
| 4 | Train |
| 5 | Loss curves |
| 6 | Generate, and watch the denoising |
| 7 | Package the results |

**Runtime → Change runtime type → L4 GPU** (A100 if available; a T4 works, about 3× slower).

## Step 0 · Get the code

In [ ]:
# @title Clone the repository and install the dependencies
import os, sys

REPO_URL = "https://github.com/a-ramakrishnan/tinystories-ARM-model.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    CLONE_DIR = "/content/tinystories-ARM-model"
    if not os.path.exists(CLONE_DIR):
        !git clone -q {REPO_URL} {CLONE_DIR}
    else:
        !git -C {CLONE_DIR} pull -q
    %cd {CLONE_DIR}
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "diffusion_models":
    %cd ..
assert os.path.exists("diffusion_models/train.py"), "Open this notebook from the tinystories-ARM-model folder."
PROJECT_DIR = os.getcwd()
print("Project folder:", PROJECT_DIR)

## Step 1 · Settings

**`run_mode`** (presets in `diffusion_models/config.py`):

| Mode | Model | Training | Time on an L4 |
|---|---|---|---|
| `smoke` | 30M (6 × 384) | 300 updates, 1M tokens | a minute: checks everything works |
| `quick` | 51M (8 × 512) | 2,000 updates, 16M tokens | a few minutes: a first look |
| `colab` | 51M (8 × 512) | 20,000 updates of 32 × 256 tokens, **164M tokens** | about 30–45 minutes |

`colab` is the run to compare: the same size, data, tokens and learning-rate schedule as our autoregressive
model *Medium 51M · TinyStories · 20k steps* (8 layers × 512, learned positions, GELU, 6e-4 → 6e-5). It uses
`torch.compile` and bfloat16 on an L4 or A100.

**`keep_in_drive`** (recommended for `colab`): the run folder (weights, training state) and a copy of the
tokenized dataset go to Google Drive, so a disconnected session can be resumed (tick `resume` and run the cells
again) and the next session skips tokenizing. Drive asks for permission once.

In [ ]:
# @title Settings
run_mode = "colab"  # @param ["smoke", "quick", "colab"]
dataset = "TinyStories"  # @param ["TinyStories", "Mini FineWeb-Edu (small)"]
keep_in_drive = True  # @param {type:"boolean"}
resume = False  # @param {type:"boolean"}
learning_rate = 0.0  # @param {type:"number"}
steps = 0  # @param {type:"integer"}
# learning_rate / steps: 0 keeps the run mode's (6e-4 peak, decaying to a tenth); set them to compare, e.g. 2e-3

import time, torch
from train_config import setup_device
from diffusion_models.config import RUN_MODES, run_mode as get_run_mode
from diffusion_models.data import FINEWEB_EDU_SMALL, TINYSTORIES

cfg, ts = get_run_mode(run_mode)
if learning_rate:
    ts.lr, ts.min_lr = learning_rate, learning_rate / 10
if steps:
    ts.steps = steps
DATASET = TINYSTORIES if dataset == "TinyStories" else FINEWEB_EDU_SMALL
dev = setup_device()
print(f"Device: {dev.device} ({dev.dtype})" + (f" · {torch.cuda.get_device_name(0)}" if dev.device == "cuda" else ""))
print(f"Run mode {run_mode!r}: {RUN_MODES[run_mode][2]}")
print(f"Peak learning rate {ts.lr:g}")
print(f"Model {cfg.n_layers} layers x {cfg.d_model}, {cfg.n_heads} heads, T = {cfg.diffusion_steps} noise levels, "
      f"context {cfg.seq_len}; {ts.steps:,} updates x {ts.batch_size} x {cfg.seq_len} = {ts.tokens * cfg.seq_len / 1e6:.0f}M tokens")

DRIVE_ROOT = None
if keep_in_drive and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/diffusion-runs"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
slug = ("tinystories" if DATASET == TINYSTORIES else "fineweb-edu-small") + f"-{run_mode}" + (f"-lr{learning_rate:g}" if learning_rate else "")
RUN_DIR = os.path.join(DRIVE_ROOT or os.path.join(PROJECT_DIR, "diffusion_models", "runs"), slug)
TOKEN_CACHE = os.path.join(DRIVE_ROOT, "tokens") if DRIVE_ROOT else None
print("Run folder:", RUN_DIR)

## Step 2 · Tokenize

GPT-2's tokenizer, as for the autoregressive models; every document is followed by `<|endoftext|>`. The first
time this downloads and tokenizes the dataset (TinyStories: about 470M tokens, several minutes on Colab's CPUs)
and, with Drive, saves the token files there; later sessions copy them back in seconds.

In [ ]:
# @title Tokenize (or copy the tokens from Drive)
import numpy as np
from diffusion_models.data import ensure_tokenized

root = ensure_tokenized(DATASET[0], DATASET[1], cache_dir=TOKEN_CACHE)
for split in ("train", "validation"):
    n = len(np.memmap(os.path.join(root, f"{split}.bin"), dtype=np.uint16, mode="r"))
    print(f"{split}: {n:,} tokens")

## Step 3 · The noising

One training window of 256 tokens at a few noise levels. At level `t`, each token is replaced by `[MASK]` (█)
with probability `t/T`; the model is trained to recover the hidden tokens from what's left, reading in both
directions. The loss counts only the masked positions.

In [ ]:
# @title Mask one window at several noise levels
from diffusion_models.data import BlockLoader
from diffusion_models.noising import corrupt_with_mask
from diffusion_models.tokens import decode

loader = BlockLoader([root], cfg.seq_len, 1, dev.device)
torch.manual_seed(0)
x = loader.batch("val")[:, :60]
for t in (8, 24, 40, 56, 64):
    noisy, _, masked = corrupt_with_mask(x, torch.tensor([t], device=x.device), cfg.diffusion_steps, cfg.mask_id)
    print(f"t = {t:2d} ({t / cfg.diffusion_steps:.0%} masked, here {masked.float().mean():.0%}):")
    print("   ", decode(noisy[0].tolist()).replace(chr(10), " "), chr(10))

## Step 4 · Train

Every `eval_interval` updates it reports two numbers on the same held-out batches:

- **masked-token loss**: cross-entropy on the masked positions (the training objective);
- **likelihood bound**: the same errors weighted by `T/t`, per token. It's an upper bound on the model's true
  loss per token, so it's the number to compare with an autoregressive model's validation loss (1.583 for the
  51M TinyStories model). The best checkpoint by the bound is kept.

With Drive, the training state is saved at every evaluation: if the session disconnects, tick `resume` in
Step 1 and run the cells again.

In [ ]:
# @title Train
from diffusion_models.train import train

summary = train(cfg, ts, [root], RUN_DIR, dev, resume=resume, dataset_names=[DATASET[0] + (f"@{DATASET[1]}" if DATASET[1] else "")])

## Step 5 · Loss curves

In [ ]:
# @title Plot
from IPython.display import Image, display
from diffusion_models.train import plot

display(Image(plot(summary, os.path.join(RUN_DIR, "loss.png"))))
print(f"Best likelihood bound: {summary['best_bound']} nats/token · {summary['tokens_per_sec']:,} tokens/s · {summary['seconds'] / 60:.0f} min")

## Step 6 · Generate

The text starts as all `[MASK]`. At each of the `T` steps the model predicts every position, and the least
confident are masked again, fewer each step, until none are left. The GIF shows every step.

In [ ]:
# @title Generate and watch the denoising
prompt = "Once upon a time"  # @param {type:"string"}
new_tokens = 128  # @param {type:"integer"}
denoising_steps = 64  # @param {type:"integer"}
top_k = 50  # @param {type:"integer"}

from diffusion_models.model import load
from diffusion_models.render import make_gif
from diffusion_models.sample import diffusion_generate

model = load(RUN_DIR, dev.device)
r = diffusion_generate(model, prompt, new_tokens, denoising_steps, top_k=top_k, seed=0)
print(r["text"])
print(f"({r['gen_len']} tokens in {r['steps']} steps, {r['seconds']} s)")
gif = make_gif(r["frames"], os.path.join(RUN_DIR, "denoising.gif"), prompt=prompt, total=r["steps"])
display(Image(filename=gif))

## Step 7 · Package the results

The weights, config, metrics (loss history, speed, samples), loss chart and GIF in one zip, downloaded to your
computer. Put it in the project folder on the Mac; integrating it into the app comes next (F3 in `docs/plan.md`).

In [ ]:
# @title Make the zip
import json, zipfile
stamp = time.strftime("%Y%m%d-%H%M")
zip_path = os.path.join(PROJECT_DIR, f"diffusion-run-{slug}-{stamp}.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for f in ("model.pt", "config.json", "metrics.json", "loss.png", "denoising.gif"):
        if os.path.exists(os.path.join(RUN_DIR, f)):
            z.write(os.path.join(RUN_DIR, f), f)
    if "r" in globals():
        z.writestr("generation.json", json.dumps({k: r[k] for k in ("prompt", "text", "steps", "gen_len", "seconds", "trace")}))
print(f"Saved {zip_path} ({os.path.getsize(zip_path) / 1e6:.0f} MB)")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)